<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  


<h1> Demo: LangChain Splitters</h1>
Developed by WeCloudData
<br></br>

In [3]:
!pip install langchain langchain-openai langchain-community langchain-chroma langchain-unstructured langchain-classic langgraph > /dev/null

In [4]:
# For the core building blocks
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.runnables import Runnable

# --- Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter

# --- Retrievers
from langchain_classic.retrievers import ParentDocumentRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# --- Vector Stores
from langchain_chroma import Chroma
from langchain_community.vectorstores import Vectara

# --- Embeddings
from langchain_openai import OpenAIEmbeddings

# --- Storage
from langchain_core.stores import InMemoryStore

# --- Loaders
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import UnstructuredFileLoader

# ---- Retrieval
from langchain_classic.chains import RetrievalQA

# --- LLMs
from langchain_openai import ChatOpenAI

# --- Other Utilities
import urllib.request

# Setup

In [ ]:
#!wget https://s3.amazonaws.com/weclouddata/datasets/genai/rag/llama2.pdf

In [5]:
url = 'https://arxiv.org/pdf/2307.09288.pdf'
file_name = 'llama2.pdf'
urllib.request.urlretrieve(url, file_name)

('llama2.pdf', <http.client.HTTPMessage at 0x7be092ae0650>)

In [6]:

def get_answer_character(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_token(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    """
    Splits documents by token and performs a RAG query.
    """
    text_splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_recursive(doc_text, chunk_size: int, chunk_overlap: int, query: str):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)
    vs = Chroma.from_documents(docs, embeddings)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vs.as_retriever())
    return qa.invoke(query)

def get_answer_parent(doc_text, query: str):
    #from langchain.retrievers.document_compressors import EmbeddingsFilter

    llm = ChatOpenAI(
        model='gpt-4.1-mini',
        temperature=0,
        openai_api_key=OPENAI_API_KEY
    )
    # This text splitter is used to create the parent documents - The big chunks
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000)
    # This text splitter is used to create the child documents - The small chunks
    # It should create documents smaller than the parent
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

    #docs = text_splitter.split_documents(doc_text)
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small",openai_api_key=OPENAI_API_KEY)

    vs = Chroma(collection_name="split_parents", embedding_function=embeddings)
    # The storage layer for the parent documents
    store = InMemoryStore()

    big_chunks_retriever = ParentDocumentRetriever(
        vectorstore=vs,
        docstore=store,
        child_splitter=child_splitter,
        parent_splitter=parent_splitter,
    )

    big_chunks_retriever.add_documents(doc_text)
    # compressor = LLMChainExtractor.from_llm(relevant_filter)
    # compression_retriever = ContextualCompressionRetriever(
    #     base_compressor=compressor, base_retriever=big_chunks_retriever,
    # )

    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=big_chunks_retriever)
    return qa.invoke(query)
    # return vs.similarity_search(query)

In [7]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(model_name = 'gpt-4.1-mini',
                 temperature=0,
                 openai_api_key=OPENAI_API_KEY)

In [8]:
%pip install --upgrade --quiet pypdf

In [9]:
from langchain_community.document_loaders import PyPDFLoader
#load llama
loader = PyPDFLoader(file_name)
#split llama
doc_text = loader.load_and_split()


In [10]:
query1 = "Is Llama2 better than GPT4?"
query2 = "What is the training hardward used for Llama2?"
query3 = "What is discuss in Figure 16: Context distillation analysis"

In [ ]:
#doc_text[33]

# Query 1: Is GPT-4 better than Llama2?

### Q1: CharacterTextSplitter

In [11]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_character(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'Is Llama2 better than GPT4?', 'result': 'Based on the information available, Llama 2 models, including the largest 70B parameter version, generally outperform many open-source models and are competitive with some closed-source models like GPT-3.5 on certain benchmarks. However, there is still a significant performance gap between Llama 2 70B and GPT-4, especially on coding benchmarks and overall capability.\n\nIn summary:\n- Llama 2 70B is close to GPT-3.5 in performance on some academic benchmarks.\n- Llama 2 70B performs on par or better than PaLM (540B) on many benchmarks.\n- GPT-4 outperforms Llama 2 70B by a notable margin across most evaluated tasks.\n\nTherefore, GPT-4 remains superior to Llama 2 in overall performance.'}


chunk=500, overlap=100, response={'query': 'Is Llama2 better than GPT4?', 'result': 'Based on the information provided, Llama 2, especially the 70B parameter model, is a very strong open-source large language model th

### Q1: TokenTextSplitter

In [12]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_token(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")


chunk=500, overlap=0, response={'query': 'Is Llama2 better than GPT4?', 'result': 'Based on the provided information, Llama 2, including the 70B parameter model, shows strong performance and outperforms many open-source models as well as some closed-source models like GPT-3.5 and PaLM (540B) on various benchmarks. However, there is still a significant performance gap between Llama 2 70B and GPT-4, with GPT-4 outperforming Llama 2 on multiple benchmarks, especially in coding tasks and other complex evaluations.\n\nIn summary, Llama 2 is a very capable open-source model and approaches or exceeds some closed-source models, but it is not better than GPT-4 according to the benchmarks and evaluations referenced.'}


chunk=500, overlap=100, response={'query': 'Is Llama2 better than GPT4?', 'result': 'Based on the information available, Llama 2, including the 70B model, outperforms many open-source models and is close to GPT-3.5 on some benchmarks like MMLU and GSM8K. However, there is still a

### Q1: RecrusiveTextSplitter

In [13]:
query = query1

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n")


chunk=500, overlap=0, response={'query': 'Is Llama2 better than GPT4?', 'result': 'Based on the available information, Llama 2 (including the 70B model) outperforms all open-source models and is close to GPT-3.5 on some benchmarks like MMLU and GSM8K. However, there is still a significant performance gap between Llama 2 70B and GPT-4. GPT-4 outperforms Llama 2 70B on various benchmarks, including coding tasks and others.\n\nSo, Llama 2 is not better than GPT-4; GPT-4 remains superior in overall performance.'}

chunk=500, overlap=100, response={'query': 'Is Llama2 better than GPT4?', 'result': 'No, Llama 2 is not better than GPT-4. According to the comparison on various academic benchmarks, Llama 2 70B performs close to GPT-3.5 and is on par or better than PaLM (540B) on most benchmarks. However, there is still a significant performance gap between Llama 2 70B and GPT-4, with GPT-4 outperforming Llama 2 by a notable margin.'}

chunk=1000, overlap=0, response={'query': 'Is Llama2 better 

# Query 2: What is the training hardward used for Llama2?

### Q2: TokenTextSplitter

In [14]:
query = query2

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_token(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The training hardware used for Llama 2 includes Meta’s Research Super Cluster and production clusters for pretraining. Specifically, the pretraining utilized a cumulative 3.3 million GPU hours of computation on hardware of type NVIDIA A100-80GB GPUs (with a TDP of 350-400W). Fine-tuning, annotation, and evaluation were performed on third-party cloud compute.'}


chunk=500, overlap=100, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The training hardware used for Llama 2 includes Meta’s Research Super Cluster and production clusters for pretraining. The specific hardware type mentioned is the A100-80GB GPUs, which have a thermal design power (TDP) of 350-400W. Fine-tuning, annotation, and evaluation were performed on third-party cloud compute.'}


chunk=1000, overlap=0, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The t

### Q2: RecrusiveTextSplitter

In [15]:
query = query2

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 200]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n")


chunk=500, overlap=0, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The training hardware used for Llama 2 includes Meta’s Research Super Cluster and production clusters for pretraining. The specific hardware type mentioned is the A100-80GB GPUs, which have a thermal design power (TDP) of 350-400W. Fine-tuning, annotation, and evaluation were performed on third-party cloud compute.'}

chunk=500, overlap=200, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The training hardware used for Llama 2 includes Meta’s Research Super Cluster and production clusters for pretraining. Fine-tuning, annotation, and evaluation were performed on third-party cloud compute. Specifically, the pretraining utilized hardware of type A100-80GB GPUs.'}

chunk=1000, overlap=0, response={'query': 'What is the training hardward used for Llama2?', 'result': 'The training hardware used for Llama 2 includes Meta’s Research Super Cluster and production

### Q3: RecrusiveTextSplitter

In [16]:
query = query3

for chunk_size in [500, 1000, 2000]:
    for chunk_overlap in [0, 100]:
        response = get_answer_recursive(doc_text, chunk_size, chunk_overlap, query)
        print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=500, overlap=0, response={'query': 'What is discuss in Figure 16: Context distillation analysis', 'result': "Figure 16 discusses the impact of context distillation based on each identified risk category. It analyzes how context distillation affects the model's performance, particularly focusing on its application to targeted samples and its influence on the RM (Reward Model) score. The figure highlights that context distillation is applied selectively—only on adversarial prompts where it improves safety—because applying it broadly can degrade response quality and increase false refusals."}


chunk=500, overlap=100, response={'query': 'What is discuss in Figure 16: Context distillation analysis', 'result': "Figure 16 discusses the impact of context distillation based on each identified risk category. It analyzes how context distillation affects the model's performance, particularly focusing on targeted samples where it increases the RM (Reward Model) score. The figure highlights t

### Q3: Parent_splitter/Child splitter

In [17]:
query = query3

response = get_answer_parent(doc_text, query)
print(f"chunk={chunk_size}, overlap={chunk_overlap}, response={response}\n\n")

chunk=2000, overlap=100, response={'query': 'What is discuss in Figure 16: Context distillation analysis', 'result': 'Figure 16: Context distillation analysis discusses the impact of context distillation on the safety reward model (RM) scores of a language model. The figure has two parts:\n\n(a) The left part shows the distribution of safety RM scores for the base model, the model with a generic preprompt, and the model with a preprompt that includes a tailored answer template based on the risk category of the prompt. It demonstrates that while adding a generic preprompt improves safety RM scores, using a preprompt with a tailored answer template leads to even greater improvements in safety.\n\n(b) The right part illustrates how context distillation affects individual samples. It shows that context distillation significantly increases the RM score for samples that initially had low safety scores, indicating improved safety responses. However, it can sometimes have a negative effect on 